# Exploratory Replay Scratchbook

In [ ]:
import sys
import pathlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "procs").exists()), pathlib.Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%matplotlib inline

from procs.gym.calibration import calibrate_as_parameters, tune_gamma
from procs.gym.data_loader import load_single_day
from procs.gym.experiment_config import ReplayExperimentConfig
from procs.gym.notebook_support import evaluate_as_fast, run_qmax_sensitivity, summarise_agent_frames
from procs.gym.helpers.plotting import plot_trajectory
from procs.gym.model_dynamics import LimitOrderModelDynamics
from procs.gym.trading_environment import TradingEnvironment
from procs.rewards import PnLReward
from procs.agents import AvellanedaStoikovAgent
from procs.stochastic_processes import MarketReplayMidpriceModel, PoissonArrivalModel, ExponentialFillFunction

In [ ]:
cfg = ReplayExperimentConfig()
DATA_PATH = cfg.data_path()
S, dt_sec, dt_index = load_single_day(str(DATA_PATH))
T_sec = float(dt_sec.sum())
sigma = MarketReplayMidpriceModel(S, dt_sec).volatility
print(f"Loaded {len(S):,} snapshots from {DATA_PATH.name}, ?={sigma:.6f}, T={T_sec:.0f}s")

In [ ]:
params = calibrate_as_parameters(
    filepath=str(DATA_PATH),
    tick_size=cfg.tick_size,
    fit_depth_max=5 * cfg.tick_size,
    plot=True,
)
sigma, A, kappa = params.sigma, params.A, params.kappa
print(f"Calibrated ?={sigma:.6f}, A={A:.4f}, ?={kappa:.0f}")

In [ ]:
as_gamma, study = tune_gamma(
    midprices=S,
    dt_array=dt_sec,
    sigma=sigma,
    kappa=kappa,
    A=A,
    tick_size=cfg.tick_size,
    Q_MAX=cfg.q_max,
    gamma_range=cfg.as_gamma_range,
    n_trials=cfg.as_gamma_trials,
    num_trajectories=cfg.evaluation_rollouts,
)
print(f"Using A-S ? = {as_gamma:.6f}")

qmax_sensitivity = run_qmax_sensitivity(
    S,
    dt_sec,
    gamma=as_gamma,
    sigma=sigma,
    kappa=kappa,
    A=A,
    terminal_time=T_sec,
    tick_size=cfg.tick_size,
    qmax_candidates=(10, 20, 50),
    num_trajectories=cfg.evaluation_rollouts,
    seed=cfg.evaluation_seed,
)
print(qmax_sensitivity.to_string(index=False, float_format="%.6f"))

In [ ]:
as_agent = AvellanedaStoikovAgent(as_gamma, sigma, kappa, T_sec, cfg.tick_size)
plot_env = TradingEnvironment(
    model_dynamics=LimitOrderModelDynamics(
        midprice_model=MarketReplayMidpriceModel(S, dt_sec, 1),
        arrival_model=PoissonArrivalModel(np.array([A, A]), 1, use_linear_approximation=False),
        fill_probability_model=ExponentialFillFunction(kappa, 1),
    ),
    reward_function=PnLReward(),
    max_inventory=cfg.q_max,
)
plot_trajectory(plot_env, as_agent, seed=cfg.evaluation_seed, datetime_index=dt_index)

seed_range = range(cfg.evaluation_seed, cfg.evaluation_seed + cfg.evaluation_rollouts)
as_eval = evaluate_as_fast(
    S,
    dt_sec,
    gamma=as_gamma,
    sigma=sigma,
    kappa=kappa,
    A=A,
    terminal_time=T_sec,
    tick_size=cfg.tick_size,
    q_max=cfg.q_max,
    seeds=seed_range,
)
print(summarise_agent_frames({"A-S Baseline": as_eval}).to_string(float_format="%.6f"))